# Timing Test — Estimating Full Inference Time
**Goal:** Run Nemotron on increasing batch sizes to estimate how long the full 500-row and full test set would take.

We test on 10, 25, and 50 rows and extrapolate from there. No need to run all 500 rows just to get a time estimate.

## 1. Imports

In [ ]:
import pandas as pd
import requests
import time
import re
import os

print("Imports OK")

## 2. Load Sample and Define Functions

In [ ]:
sample = pd.read_csv("../data/personas_sample_500.csv")

# Filter bad rows
sample = sample[
    ~sample["occupation"].str.lower().str.strip().isin([
        "no occupation", "not in workforce", "not_in_workforce", ""
    ])
].reset_index(drop=True)

print(f"Clean sample: {len(sample)} rows")
print(sample["label_name"].value_counts())

In [ ]:
OLLAMA_URL = "http://localhost:11434/v1/chat/completions"
MODEL_NAME = "nemotron-mini"

SYSTEM_PROMPT = """You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college."""


def serialize_row(row):
    return (
        f"A {int(row['age'])}-year-old {str(row['sex']).lower().strip()}, "
        f"{str(row['marital_status']).replace('_', ' ').strip()}, "
        f"working as a {str(row['occupation']).replace('_', ' ').strip()}. "
        f"Located in {str(row['state']).strip()}."
    )


def build_zero_shot_prompt(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Classify this person's education level:\n\n"
            f"{serialize_row(row)}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


def parse_response(content):
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    trace = think_match.group(1).strip() if think_match else ""
    label_match = re.search(r"\b(not_college|college)\b", content, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else "UNKNOWN"
    return label, trace


def classify_row(row):
    messages = build_zero_shot_prompt(row)
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "max_tokens": 512,
        "temperature": 0.1,
        "stream": False,
    }
    start = time.time()
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=120).json()
        elapsed = time.time() - start
        content = response["choices"][0]["message"]["content"]
        label, trace = parse_response(content)
    except Exception as e:
        elapsed = time.time() - start
        label, trace = "ERROR", str(e)
    return label, round(elapsed * 1000)


print("Functions defined.")

## 3. Timing Test — 10 rows

In [ ]:
batch_10 = sample.sample(10, random_state=42).reset_index(drop=True)
times_10 = []

print("Running 10 rows...")
t_start = time.time()

for i, row in batch_10.iterrows():
    label, ms = classify_row(row.to_dict())
    times_10.append(ms)
    print(f"  Row {i+1:2d} | {ms}ms | {label}")

total_10 = time.time() - t_start
avg_ms   = sum(times_10) / len(times_10)

print(f"\n── 10-row results ──")
print(f"Total time:      {total_10:.1f}s")
print(f"Avg time/row:    {avg_ms:.0f}ms")
print(f"Min / Max:       {min(times_10)}ms / {max(times_10)}ms")

## 4. Timing Test — 25 rows

In [ ]:
batch_25 = sample.sample(25, random_state=42).reset_index(drop=True)
times_25 = []

print("Running 25 rows...")
t_start = time.time()

for i, row in batch_25.iterrows():
    label, ms = classify_row(row.to_dict())
    times_25.append(ms)
    print(f"  Row {i+1:2d} | {ms}ms | {label}")

total_25 = time.time() - t_start
avg_ms_25 = sum(times_25) / len(times_25)

print(f"\n── 25-row results ──")
print(f"Total time:      {total_25:.1f}s")
print(f"Avg time/row:    {avg_ms_25:.0f}ms")
print(f"Min / Max:       {min(times_25)}ms / {max(times_25)}ms")

## 5. Timing Test — 50 rows

In [ ]:
batch_50 = sample.sample(50, random_state=42).reset_index(drop=True)
times_50 = []

print("Running 50 rows...")
t_start = time.time()

for i, row in batch_50.iterrows():
    label, ms = classify_row(row.to_dict())
    times_50.append(ms)
    print(f"  Row {i+1:2d} | {ms}ms | {label}")

total_50 = time.time() - t_start
avg_ms_50 = sum(times_50) / len(times_50)

print(f"\n── 50-row results ──")
print(f"Total time:      {total_50:.1f}s")
print(f"Avg time/row:    {avg_ms_50:.0f}ms")
print(f"Min / Max:       {min(times_50)}ms / {max(times_50)}ms")

## 6. Extrapolation — How Long Will the Full Run Take?

In [ ]:
# Use the 50-row average as the most reliable estimate
avg_ms = avg_ms_50

estimates = {
    "500 rows (Week 3 experiment)": 500,
    "Full 25,910 test rows":        25910,
    "Full 1M dataset":              1_000_000,
}

print(f"Based on average of {avg_ms:.0f}ms per row:\n")
print(f"{'Batch':<35} {'Time':>12}")
print("-" * 50)

for label, n_rows in estimates.items():
    total_sec  = (avg_ms / 1000) * n_rows
    total_min  = total_sec / 60
    total_hr   = total_min / 60

    if total_hr >= 1:
        time_str = f"{total_hr:.1f} hours"
    elif total_min >= 1:
        time_str = f"{total_min:.1f} minutes"
    else:
        time_str = f"{total_sec:.1f} seconds"

    print(f"{label:<35} {time_str:>12}")

print(f"\nConclusion:")
mins_500 = (avg_ms / 1000) * 500 / 60
print(f"  500 rows will take ~{mins_500:.0f} minutes on your laptop")
print(f"  This is why we don't run the full 25K test set with an LLM")

## 7. Summary Table

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"batch": "10 rows",  "total_s": round(total_10, 1), "avg_ms": round(avg_ms,    0)},
    {"batch": "25 rows",  "total_s": round(total_25, 1), "avg_ms": round(avg_ms_25, 0)},
    {"batch": "50 rows",  "total_s": round(total_50, 1), "avg_ms": round(avg_ms_50, 0)},
])

print("=== TIMING SUMMARY ===")
print(summary.to_string(index=False))

os.makedirs("../results", exist_ok=True)
summary.to_csv("../results/timing_test.csv", index=False)
print("\nSaved: results/timing_test.csv")